# VGG-16: feature extractor y fine-tuning (CIFAR-10)

## Configuración y datos

In [6]:
%pip install certifi
import os, certifi
os.environ["SSL_CERT_FILE"] = certifi.where()


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import copy
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import vgg16, VGG16_Weights
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"torch {torch.__version__} | device: {device}")
Path("models").mkdir(exist_ok=True)

torch 2.14.1 | device: mps


In [2]:
train_raw = datasets.CIFAR10(root="data", train=True, download=True)
y_train = np.array(train_raw.targets)
CLASSES = train_raw.classes

# Mismo split 45,000 / 5,000 que el resto del equipo 
if Path("split_indices.npz").exists():
    s = np.load("split_indices.npz")
    idx_train, idx_val = s["train"], s["val"]
    idx_train_small = s["train_small"] if "train_small" in s.files else None
else:
    idx_train, idx_val = train_test_split(
        np.arange(len(y_train)), test_size=5_000, stratify=y_train, random_state=SEED
    )
    idx_train_small = None
print(f"train: {len(idx_train):,} | val: {len(idx_val):,}")

VGG_RES = 128   # misma resolución para feature extractor y fine-tuning 
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
norm = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
resize = transforms.Resize((VGG_RES, VGG_RES))

eval_tf = transforms.Compose([resize, transforms.ToTensor(), norm])
train_tf_aug = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(p=0.5),
    resize, transforms.ToTensor(), norm,
])

def make_datasets(train_tf):
    full_train = datasets.CIFAR10("data", train=True, download=False, transform=train_tf)
    full_eval = datasets.CIFAR10("data", train=True, download=False, transform=eval_tf)
    return {"train": Subset(full_train, idx_train), "val": Subset(full_eval, idx_val)}

DSETS = {True: make_datasets(train_tf_aug), False: make_datasets(eval_tf)} 

def make_loaders(augment, batch_size):
    d = DSETS[augment]
    return (
        DataLoader(d["train"], batch_size=batch_size, shuffle=True, num_workers=0),
        DataLoader(d["val"], batch_size=128, shuffle=False, num_workers=0),
    )

100%|██████████| 170M/170M [00:34<00:00, 4.94MB/s] 


train: 45,000 | val: 5,000


Pasar de 32×32 a 128×128 multiplica los píxeles por 16, y el costo de las convoluciones escala casi linealmente con los píxeles. A 128 px el backbone produce un mapa de 512×4×4; avgpool lo reescala a 7×7 para que las dimensiones del clasificador preentrenado sigan siendo válidas. 

## Construcción de los modelos
VGG-16 con pesos IMAGENET1K_V1. model.features agrupa las 13 convoluciones en 5 bloques; cada bloque termina en un MaxPool2d. La última capa del clasificador se reemplaza por Linear(4096, 10).

| Bloque | Índices en model.features |
|---|---|
| 1 | 0–4 |
| 2 | 5–9 |
| 3 | 10–16 |
| 4 | 17–23 |
| 5 | 24–30 |

In [3]:
BLOCK_RANGES = {1: (0, 5), 2: (5, 10), 3: (10, 17), 4: (17, 24), 5: (24, 31)}  

def build_vgg16(unfrozen=(), n_classes=10):
    """VGG-16 preentrenada. unfrozen=() -> feature extractor; unfrozen=(5,) o (4, 5) -> fine-tuning.
    El clasificador siempre se entrena."""
    model = vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
    model.classifier[6] = nn.Linear(4096, n_classes)      # nueva capa de salida 
    for p in model.parameters():                          # congelar todo
        p.requires_grad = False
    for b in unfrozen:                                    # descongelar bloques elegidos
        lo, hi = BLOCK_RANGES[b]
        for p in model.features[lo:hi].parameters():
            p.requires_grad = True
    for p in model.classifier.parameters():               # el clasificador siempre se entrena
        p.requires_grad = True
    return model

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def block_params(model):
    return {b: sum(p.numel() for p in model.features[lo:hi].parameters())
            for b, (lo, hi) in BLOCK_RANGES.items()}

m = build_vgg16(unfrozen=())
total, trainable = count_params(m)
print(f"Feature extractor -> totales: {total:,} | entrenables: {trainable:,}")
print("Parámetros por bloque:", block_params(m))
print("Clasificador:", m.classifier)
assert total == 134_301_514 and trainable == 119_586_826   # valores esperados de VGG-16 con cabeza de 10 clases

m = build_vgg16(unfrozen=(4, 5))
print("Fine-tuning bloques 4 y 5 -> entrenables:", f"{count_params(m)[1]:,}")
del m

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /Users/Camila/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [01:21<00:00, 6.79MB/s] 


Feature extractor -> totales: 134,301,514 | entrenables: 119,586,826
Parámetros por bloque: {1: 38720, 2: 221440, 3: 1475328, 4: 5899776, 5: 7079424}
Clasificador: Sequential(
  (0): Linear(in_features=25088, out_features=4096, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=4096, out_features=4096, bias=True)
  (4): ReLU(inplace=True)
  (5): Dropout(p=0.5, inplace=False)
  (6): Linear(in_features=4096, out_features=10, bias=True)
)
Fine-tuning bloques 4 y 5 -> entrenables: 132,566,026
